# EduCast backend — Google Colab only

teachgen **只在 Colab 上跑**，不在本机跑 Python。

## 准备（一次性）
1. 把整个 `education_video` 上传到 Google Drive（**`Teaching_Video_Generation` 保持你最初的版本，不用改**）
2. 只需更新 Drive 里这几个**外层**文件：`website_run.ipynb`、`website_server.py`、`run_teachgen.py`、`api_usage_tracker.py`
3. 注册 [ngrok](https://ngrok.com/) 免费账号，复制 **Authtoken**
4. 前端用 Lovable 或本机 `npm run dev`（只跑 UI）

## 运行顺序
从上到下 **依次运行每个 cell**。最后一个 cell 会保持运行，不要停止。
把打印出来的 **Backend URL** 填到前端页面的 Backend URL 输入框。

## 前端报 Failed to fetch？
1. Backend URL 必须是 Cell 3 打印的 `https://xxxx.ngrok-free.dev`（不是 authtoken 页面）
2. 先在浏览器打开 `{Backend URL}/health`，应看到 JSON
3. Cell 3 和 Cell 4 都要保持运行

## 日志说明
- `ALSA lib ...`：**可忽略**
- `[usage] est. $0.0123 | tokens ...`：**实时 API 费用估算**（前端也会显示）
- 完整 pipeline 含 **Manim animation**；`run_teachgen.py` 负责 Colab 兼容，**不修改** `Teaching_Video_Generation` 源码

In [ ]:
# ── 1. 挂载 Drive，进入项目目录 ─────────────────────────────────────────
from google.colab import drive
from pathlib import Path
import os
import sys

drive.mount("/content/drive")

# 改成你在 Drive 里的实际路径
PROJECT_DIR = Path("/content/drive/MyDrive/education_video")

if not (PROJECT_DIR / "website_server.py").exists():
    raise FileNotFoundError(
        f"找不到 website_server.py，请检查 PROJECT_DIR。\n当前: {PROJECT_DIR}"
    )
if not (PROJECT_DIR / "run_teachgen.py").exists():
    raise FileNotFoundError(
        f"找不到 run_teachgen.py，请把该文件上传到 Drive（与 website_server.py 同级）。\n当前: {PROJECT_DIR}"
    )
if not (PROJECT_DIR / "api_usage_tracker.py").exists():
    raise FileNotFoundError(
        f"找不到 api_usage_tracker.py，请上传到 Drive（与 website_server.py 同级）。\n当前: {PROJECT_DIR}"
    )

os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

os.environ["WEBSITE_HOST"] = "0.0.0.0"
os.environ["WEBSITE_PORT"] = "8765"
os.environ.setdefault("MANIM_RENDERER", "cairo")
os.environ["TEACHGEN_MAX_WORKERS"] = "20"  # teachgen 段级并行

print("项目目录:", PROJECT_DIR)
print("teachgen:", (PROJECT_DIR / "Teaching_Video_Generation").exists())

In [ ]:
# ── 2. 安装系统与 Python 依赖（含 Manim / code2video）────────────────────
!apt-get update -qq
!apt-get install -y -qq ffmpeg \
    libcairo2-dev libpango1.0-dev pkg-config libffi-dev \
    texlive texlive-latex-extra texlive-fonts-extra tipa dvisvgm \
    > /dev/null

!pip install -q fastapi uvicorn python-multipart pyngrok openai python-dotenv
!pip install -q -r Teaching_Video_Generation/requirements.txt
!pip install -q -r Teaching_Video_Generation/code2video/requirements.txt

import shutil
import subprocess

print("ffmpeg:", shutil.which("ffmpeg"))
subprocess.run(["manim", "--version"], check=False)

In [ ]:
# ── 3. ngrok 公网隧道（前端通过此 URL 访问 Colab 后端）────────────────────
from getpass import getpass
import time
import urllib.request

import website_server
from pyngrok import ngrok

PORT = int(os.environ.get("WEBSITE_PORT", "8765"))

if not os.environ.get("NGROK_AUTHTOKEN"):
    os.environ["NGROK_AUTHTOKEN"] = getpass(
        "ngrok Authtoken (https://dashboard.ngrok.com/get-started/your-authtoken): "
    ).strip()

website_server.start_background()
time.sleep(3)

tunnel = ngrok.connect(PORT, "http")
PUBLIC_URL = tunnel.public_url.rstrip("/")
os.environ["WEBSITE_PUBLIC_URL"] = PUBLIC_URL

for _ in range(10):
    try:
        req = urllib.request.Request(
            f"{PUBLIC_URL}/health",
            headers={"ngrok-skip-browser-warning": "1"},
        )
        with urllib.request.urlopen(req, timeout=5) as resp:
            print(resp.read().decode())
        break
    except Exception:
        time.sleep(1)

print("\n" + "=" * 60)
print("Backend URL（复制到前端 Backend URL 输入框）:")
print(PUBLIC_URL)
print("=" * 60)

In [ ]:
# ── 4. 保持 Colab 会话（不要停止此 cell）────────────────────────────────
import time

print("后端运行中。去前端填入 Backend URL，粘贴 API key + JSON，点击 Generate。")
print("停止此 cell = 关闭后端。")
while True:
    time.sleep(3600)